## ⚡ CTAS (Create Table As Select)

Hasta ahora hemos aprendido a crear tablas definiendo manualmente su estructura mediante la sentencia `CREATE TABLE`.

Sin embargo, en escenarios reales de Data Engineering existe una forma mucho más rápida y práctica de crear una tabla.

### 🤔 ¿Y si la estructura ya existe?

Imaginemos que ya disponemos de una consulta SQL con toda la información que necesitamos.

¿Tiene sentido volver a definir cada columna manualmente?

La respuesta es **no**.

Para estos casos existen los **CTAS (Create Table As Select)**.

---

### 🚀 ¿Qué es un CTAS?

Un **CTAS** permite crear una nueva tabla directamente a partir del resultado de una consulta SQL.

Durante este proceso, Databricks:

* ✅ Ejecuta la consulta
* ✅ Infiere automáticamente el esquema resultante
* ✅ Crea la nueva tabla
* ✅ Inserta los datos en una sola operación

Esto simplifica considerablemente la creación de tablas derivadas para procesos ETL, análisis y capas del Lakehouse.

---

#### 📖 Sintaxis General

```sql id="l18ymk"
CREATE TABLE nombre_catalago.nombre_esquema.nombre_nueva_tabla
USING formato
COMMENT 'Descripción de la tabla'
PARTITION BY (columna)
LOCATION 'ruta_física'
AS

SELECT ...
FROM nombre_catalago.nombre_esquema.tabla_origen;
```

---

##### 🔍 ¿Qué hace cada propiedad?

* **📦 USING**

    Define el formato físico con el que se almacenarán los datos de la nueva tabla.

    Ejemplo:

    * DELTA
    * PARQUET
    * ICEBERG (cuando aplique)

* **📝 COMMENT**

    * Permite agregar una descripción o documentación a la tabla.
    * Esta información resulta útil para facilitar la comprensión y el gobierno de los datos.

* **📂 PARTITION BY**

    * Especifica la columna utilizada para particionar físicamente la tabla.
    * Una estrategia adecuada de particionamiento puede mejorar significativamente el rendimiento de determinadas consultas.

* **📍 LOCATION**

    * Define la ubicación física donde se almacenarán los archivos de la nueva tabla.
    * Esta propiedad también determina el tipo de tabla que se creará.
    * ✅ Si **no** especificamos `LOCATION`, Databricks almacenará los datos en la ubicación administrada por Unity Catalog, creando una **Managed Table**.
    * ✅ Si especificamos `LOCATION`, los datos se almacenarán en la ruta indicada, creando una **External Table**, siempre que dicha ubicación esté registrada mediante un **External Location** y un **Storage Credential**.

---

### 💡 ¿Cuándo utilizar CTAS?

CTAS es una excelente opción cuando necesitamos:

* ✅ Crear tablas derivadas para procesos ETL.
* ✅ Materializar el resultado de consultas complejas.
* ✅ Generar capas Bronze, Silver o Gold.
* ✅ Evitar definir manualmente el esquema de una tabla ya existente.

En muchos proyectos de Data Engineering, CTAS es una de las formas más rápidas y utilizadas para construir nuevas tablas dentro del Lakehouse.


### 🚀 Punto de Inicio en Databricks

Antes de trabajar con Delta Lake necesitamos una sesión de Spark activa.

Spark será el motor encargado de:

* ✅ Leer datos
* ✅ Transformarlos
* ✅ Procesarlos de forma distribuida
* ✅ Persistirlos como Delta Tables

In [0]:
from pyspark.sql import SparkSession # Puerta de entrada para trabajar con spark <-- SIEMPRE DEBEMOS IMPORTAR LA LLAVE MAESTRA QUE INICIA TODO.
from pyspark.sql.functions import *  # Funciones propias del módulo SQL de Spark, para trabajar sobre Dataframes.
spark = SparkSession.builder.appName("08CTAs").getOrCreate() 
"""
^          ^__________^        ^_________^                               ^
|                |                   |                                   | 
Variable   Constructor de Sesión   Nombre Aplicación       Evita conflicto del SparkSession"""

print("🚀 Spark Session iniciada correctamente")

#### 🔍 ¿Qué acaba de ocurrir?

Acabamos de crear una Spark Session. Esta sesión representa nuestro punto de entrada hacia:
* 🏗️ Apache Spark
* 🏗️ Databricks Runtime
* 🏗️ Delta Lake
* 🏗️ Unity Catalog

### ⚡ CTAs - Managed Table

- Nombre: `catalog_databricks_2026_de.schema_databricks_2026_de.customers_delta_table_managed_cta`

In [0]:
## CREACIÓN DE MANAGED TABLE

#### PASO 1. DEFINIR QUERY SQL (TABLA ORIGEN)

# display(spark.sql("SELECT * FROM catalog_databricks_2026_de.schema_databricks_2026_de.customers_delta_table"))

#### PASO 2. ADJUNTAR QUERY SQL A CTAs

spark.sql("""
          
          CREATE TABLE catalog_databricks_2026_de.schema_databricks_2026_de.customers_delta_table_managed_cta
          USING DELTA
          COMMENT 'Managed Table Customers - CTAs'
          AS
          SELECT * 
          FROM 
          catalog_databricks_2026_de.schema_databricks_2026_de.customers_delta_table

          """)
print("Managed Table creada a partir de CTAs")

### ⚡ CTAs - External Table

- Nombre: `catalog_databricks_2026_de.schema_databricks_2026_de.customers_delta_external_table_cta`

In [0]:
## CREACIÓN DE EXTERNAL TABLE

#### PASO 1. DEFINIR QUERY SQL (TABLA ORIGEN)

# display(spark.sql("SELECT * FROM catalog_databricks_2026_de.schema_databricks_2026_de.customers_delta_table"))

#### PASO 2. ADJUNTAR QUERY SQL A CTAs

spark.sql("""
          
          CREATE TABLE catalog_databricks_2026_de.schema_databricks_2026_de.customers_delta_external_table_cta
          USING CSV
          COMMENT 'Managed Table Customers - CTAs'
          LOCATION 's3://bucket-datasets-brayan/external_table/'
          AS
          SELECT * 
          FROM 
          catalog_databricks_2026_de.schema_databricks_2026_de.customers_delta_table

          """)
print("External Table creada a partir de CTAs")

#### Resultado External Table en S3
![Resultado](assets/Resultado_CTA_External_Table.png)